# Secondary School Retention Risk Analytics
## Prioritizing fee support before students leave school

**Analyst:** Jesyldah  
**Organization:** ElimuMatch (Kenya secondary education)  
**Dataset:** Synthetic student cohort, n = 1,000 (seed 2026)

---

### Executive summary

Kenyan secondary enrolment has expanded, but many students still leave before completing the year because of fees, health, distance, and academic pressure. ElimuMatch helps schools and sponsors **prioritize support** when resources are limited, instead of relying on visibility or personal networks.

This notebook walks through the analytics behind that ranking: synthetic data generation, exploration, modeling, fairness checks, and intervention sizing. Staff review the ranked list before any fee gift or referral goes out.

**Objective:** Rank secondary students by retention risk so schools and sponsors can target fee support and related interventions before dropout occurs. Fee support is one channel; health, commute, and academic factors are also modeled.

**Approach:** Model retention from academic, health, access, and socioeconomic signals. Fee arrears sit in the gift ledger, not as the sole predictor. When models are comparable on AUC, prefer the one with higher dropout recall; logistic regression is selected over gradient boosting in that case because it is easier to explain to staff.

**Data:** Synthetic cohort (n = 1,000, seed 2026). Results validate pipeline design and assumptions, not live national performance. Partner school data would replace this cohort before any public ranking is published.


## 1. Problem statement

Secondary enrolment in Kenya has expanded, but many students still leave before completing the year because of fees, health, distance, and academic pressure. Support is often allocated by visibility or personal networks rather than measured risk.

**Business question:** Which students are most likely to drop out, and can they be ranked early enough to target fee gifts and other support?

**Analytics requirements**
- Rank students for staff review (not full automation)
- Favor models that detect leavers, not only high overall accuracy
- Exclude leakage fields and post-outcome variables from training
- Support explainability for operations staff
- Monitor performance by socioeconomic group


## 2. Analysis workflow

| Step | Task |
|---|---|
| 1 | Environment setup |
| 2 | Synthetic data generation (documented DGP) |
| 3 | Profile cohort |
| 4 | Data quality checks |
| 5-6 | Exploratory analysis |
| 7 | Feature engineering |
| 8 | Preprocessing (train/test split, imputation) |
| 9-10 | Model training, comparison, and selection |
| 11-12 | Evaluation (ROC, confusion matrix, fairness) |
| 13-14 | Explainability and intervention queue |
| 15 | Conclusion and recommendation |


### Step 1: Environment setup

*Before you run:* use the **`analysis_notebook/`** folder (portable bundle with only what this notebook needs). Clone or download that folder into your working directory, or unzip it if shared as a zip. You do not need the full ElimuMatch repository.

```bash
cd analysis_notebook
pip install -r requirements.txt
jupyter notebook ElimuMatch_Analysis.ipynb
```

The bundle includes `feature_engineering.py`, `preprocess_data.py`, and `kenya_schools.py`. *The cell below locates that folder and adds it to `sys.path` so imports work.*


#### Code


In [ ]:
# Resolve project root so local .py modules import reliably
from pathlib import Path
import sys

def find_project_root() -> Path:
    markers = ("feature_engineering.py", "preprocess_data.py", "kenya_schools.py")
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if all((candidate / name).exists() for name in markers):
            return candidate
    raise FileNotFoundError(
        "analysis_notebook bundle not found. Open this notebook from the analysis_notebook/ "
        "folder, or download that folder from the ElimuMatch repo."
    )

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    RocCurveDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

from feature_engineering import ENGINEERED_FEATURES, engineer_features, correlation_with_target
from preprocess_data import LEAKAGE_COLUMNS, MISSINGNESS_COLS, TARGET, preprocess

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

DATA_FILE = ROOT / "elimu_match_data_v4.csv"
RANDOM_STATE = 2026
DROPOUT_LABEL = 0  # in our data, retained=0 means the student dropped out

print(f"Project root: {ROOT}")
print(f"Saved cohort on disk (optional until Step 3/8): {DATA_FILE.exists()}")


#### Findings

Imports should succeed and `Project root` should point at the folder that contains `feature_engineering.py` and the other bundle modules. `RANDOM_STATE = 2026` is used for generation and for the train/test split in Step 8.


### Step 2: Synthetic data generation

Partner student records were not available for this project. **This section is included to show how the cohort is generated**: the logic, assumptions, and reasoning behind each variable (see comments in the code cell). The implementation mirrors `synthetic_data_v2.py` and builds `df` in memory for the steps that follow.

*Default path: run this step. Skip this step only if you already have `elimu_match_data_v4.csv` on disk and will load it in Step 3.*


#### Code


In [ ]:
# Synthetic cohort generation (synthetic_data_v2.py logic, seed 2026)
# Partner records were unavailable — this DGP creates a realistic but fully synthetic cohort.
from kenya_schools import N_SCHOOLS

rng = np.random.default_rng(RANDOM_STATE)
n_students = 1000
n_schools = N_SCHOOLS  # 47 counties — one sample school per county in the catalog

DROPOUT_REASONS = [
    "financial_instability", "academic_performance", "health_related",
    "commute_distance", "psychosocial", "household_pressure", "other",
]
MISSINGNESS_CONFIG = {
    "cash_flow_volatility": 0.08,
    "commute_barrier_score": 0.06,
    "digital_equity_access_score": 0.06,
    "psychosocial_support_access": 0.07,
}


def _zscore(series):
    std = series.std()
    return np.zeros_like(series, dtype=float) if std == 0 else (series - series.mean()) / std


def _inject_missing(values, missing_rate, ses_index, rng):
    # Lower-SES students are slightly more likely to have missing survey fields (MAR).
    ses_weight = (6 - ses_index) / 5
    prob = np.clip(missing_rate * (0.75 + 0.50 * ses_weight), 0, 0.20)
    mask = rng.random(len(values)) < prob
    out = values.astype(object).copy()
    out[mask] = np.nan
    return out


def _assign_dropout_reasons(frame, rng):
    # Post-outcome label for simulation only — never used as a model feature.
    reasons = np.full(len(frame), np.nan, dtype=object)
    risk_matrix = np.column_stack([
        _zscore(frame["cash_flow_volatility"].fillna(frame["cash_flow_volatility"].median()).values)
        + _zscore((6 - frame["socioeconomic_status_index"]).values),
        _zscore(frame["failed_subjects_count"].values) - _zscore(frame["gpa_trend"].values),
        _zscore(frame["health_related_absences"].values) + _zscore(frame["chronic_health_risk_score"].values),
        _zscore(frame["commute_barrier_score"].fillna(frame["commute_barrier_score"].median()).values),
        -_zscore(frame["social_integration_score"].values)
        - _zscore(frame["psychosocial_support_access"].fillna(0).values),
        _zscore(frame["resource_dilution_index"].values) + _zscore((6 - frame["socioeconomic_status_index"]).values),
    ])
    risk_matrix += rng.normal(0, 0.15, risk_matrix.shape)
    dropped_idx = np.where(frame["retained"].values == 0)[0]
    reason_idx = risk_matrix[dropped_idx].argmax(axis=1)
    for i, row_idx in enumerate(dropped_idx):
        reasons[row_idx] = "other" if rng.random() < 0.08 else DROPOUT_REASONS[reason_idx[i]]
    return reasons


# --- Identifiers ---
# student_id: Sequential integer 1..n. Deterministic primary key for joins and ledger tables.
student_id = np.arange(1, n_students + 1)

# school_id: Random assignment across the national school catalog (1..47).
# Discrete uniform over schools — national design, not one-site concentration.
school_id = rng.integers(1, n_schools + 1, n_students)

# --- Base demographics ---
# age_at_enrollment: Secondary students typically age 13–17.
# Discrete uniform on [13, 17] — standard Form 1–4 age band.
age = rng.integers(13, 18, n_students)

# gender: Binary indicator (0/1) for modeling and fairness slices.
# Bernoulli with p=0.5 — balanced split in the synthetic cohort.
gender = rng.integers(0, 2, n_students)

# socioeconomic_status_index: Income/resource quintile proxy (1=most constrained, 5=least).
# Discrete distribution skewed toward lower quintiles to reflect sector equity pressure.
ses_index = rng.choice([1, 2, 3, 4, 5], size=n_students, p=[0.25, 0.25, 0.22, 0.18, 0.10])

# resource_dilution_index: Household size / crowding pressure.
# Poisson with mean rising as SES falls — larger households at lower quintiles.
resource_dilution = np.array([rng.poisson(lam=3.5 + 0.35 * (6 - ses)) for ses in ses_index])

# --- Institutional and environmental access ---
# digital_equity_access_score: 0=no reliable access, 1=shared, 2=personal device.
# SES-conditional categorical probabilities — better access at higher quintiles.
digital_probs = {
    1: [0.72, 0.20, 0.08], 2: [0.60, 0.28, 0.12], 3: [0.48, 0.32, 0.20],
    4: [0.35, 0.38, 0.27], 5: [0.22, 0.40, 0.38],
}
digital_equity = np.array([rng.choice([0, 1, 2], p=digital_probs[ses]) for ses in ses_index])

# nutritional_support_access: School feeding program available (0/1).
# Binomial with probability increasing as SES decreases — feeding targets needier students.
nutritional_support = np.array([
    rng.binomial(1, p=min(0.85, 0.40 + 0.10 * (6 - ses))) for ses in ses_index
])

# psychosocial_support_access: Counseling / psychosocial program access (0/1).
# Binomial with modest coverage rising slightly with SES (better-resourced schools).
psychosocial_support = np.array([rng.binomial(1, p=0.12 + 0.06 * ses) for ses in ses_index])

# commute_barrier_score: Distance / travel burden proxy in km-like units.
# Gamma distribution — many short commutes, long tail for rural day-school friction.
commute_barrier = np.array([rng.gamma(shape=2, scale=2.0 + 0.45 * (6 - ses)) for ses in ses_index])

# --- Academic and health ---
# chronic_health_risk_score: Underlying health intensity (1=low, 2=medium, 3=high).
# Categorical PMF: 70% low, 20% medium, 10% high — most students not chronically ill.
chronic_health = rng.choice([1, 2, 3], size=n_students, p=[0.7, 0.2, 0.1])

# health_related_absences: School days missed for health reasons.
# Poisson with rate scaled by chronic_health — worse health → more absences.
health_absences = rng.poisson(lam=chronic_health * 5, size=n_students)

# failed_subjects_count: Number of failed core subjects.
# Poisson with mean rising with lower SES and higher health risk.
failed_subjects = np.array([
    rng.poisson(lam=0.7 + 0.30 * (6 - ses) + 0.15 * (health - 1))
    for ses, health in zip(ses_index, chronic_health)
])

# gpa_trend: Year-on-year GPA change (negative = decline).
# Linear function of failures and SES plus Gaussian noise — academic momentum signal.
gpa_trend = (
    -0.55 * failed_subjects
    + 0.25 * (ses_index - 3)
    + rng.normal(0, 0.6, n_students)
)

# strength_science_indicator: STEM aptitude flag (0/1).
# Binomial — higher SES and fewer failures raise probability of science strength.
science_talent = np.array([
    rng.binomial(1, p=min(0.65, max(0.08, 0.12 + 0.05 * ses - 0.10 * failed)))
    for ses, failed in zip(ses_index, failed_subjects)
])

# social_integration_score: Extracurricular / belonging proxy (0–3).
# Normal draw clipped to range — higher SES and fewer absences improve integration.
social_integration = np.array([
    int(np.clip(rng.normal(1.2 + 0.15 * ses - 0.04 * absences, 0.9), 0, 3))
    for ses, absences in zip(ses_index, health_absences)
])

# --- Economic volatility ---
# cash_flow_volatility: Share of income variance from irregular sources (e.g. agriculture).
# Uniform band widens as SES falls — 0.12–0.32 after clipping.
cash_flow_volatility = np.array([
    rng.uniform(0.12 + 0.02 * (6 - ses), 0.22 + 0.03 * (6 - ses)) for ses in ses_index
])

# academic_catchup_status: Remediation flag (0/1).
# Deterministic rule: >3 failed subjects → catch-up required (dropped from model as redundant).
academic_catchup = np.where(failed_subjects > 3, 1, 0)

# --- School-level random effect ---
# Small random shift per school_id to mimic unobserved school context (not a modeled feature).
school_effect = {sid: rng.normal(0, 0.25) for sid in range(1, n_schools + 1)}
school_shift = np.array([school_effect[sid] for sid in school_id])

# --- Retention outcome (target) ---
# retention_risk_score: Latent probability from a logistic structural model (oracle for DGP only).
# Coefficients tuned so cohort retention ≈ 86–87%. Excluded from training (leakage).
retention_logit = (
    3.05
    + 0.22 * ses_index
    + 0.30 * gpa_trend
    - 0.09 * commute_barrier
    - 0.05 * health_absences
    - 0.28 * failed_subjects
    + 0.35 * nutritional_support
    + 0.30 * psychosocial_support
    + 0.18 * digital_equity
    + 0.20 * science_talent
    + 0.12 * social_integration
    - 1.5 * (cash_flow_volatility - 0.22)
    - 0.06 * (resource_dilution - 4)
    + school_shift
    + rng.normal(0, 0.35, n_students)
)
retention_risk_score = 1 / (1 + np.exp(-retention_logit))

# retained: Binary outcome (1=stayed enrolled, 0=dropped out).
# Bernoulli draw using retention_risk_score as probability — stochastic final outcome.
retained = rng.binomial(1, retention_risk_score)

df = pd.DataFrame({
    "student_id": student_id,
    "school_id": school_id,
    "age_at_enrollment": age,
    "gender": gender,
    "resource_dilution_index": resource_dilution,
    "socioeconomic_status_index": ses_index,
    "commute_barrier_score": commute_barrier,
    "digital_equity_access_score": digital_equity,
    "nutritional_support_access": nutritional_support,
    "gpa_trend": gpa_trend,
    "failed_subjects_count": failed_subjects,
    "strength_science_indicator": science_talent,
    "chronic_health_risk_score": chronic_health,
    "health_related_absences": health_absences,
    "social_integration_score": social_integration,
    "cash_flow_volatility": cash_flow_volatility,
    "academic_catchup_status": academic_catchup,
    "psychosocial_support_access": psychosocial_support,
    "retention_risk_score": retention_risk_score.round(4),
    "retained": retained,
})

# Clip to realistic bounds used in validation (descriptive_analysis.py ranges).
df["commute_barrier_score"] = df["commute_barrier_score"].clip(0, 20)
df["gpa_trend"] = df["gpa_trend"].clip(-4, 4)
df["failed_subjects_count"] = df["failed_subjects_count"].clip(0, 8)
df["cash_flow_volatility"] = df["cash_flow_volatility"].clip(0.12, 0.32)

# dropout_reason: Assigned only for retained==0 from dominant risk dimension (+ 8% "other").
df["dropout_reason"] = _assign_dropout_reasons(df, rng)

# Inject missing values on survey-like fields (rates in MISSINGNESS_CONFIG).
for column, rate in MISSINGNESS_CONFIG.items():
    df[column] = _inject_missing(
        df[column].values, rate, df["socioeconomic_status_index"].values, rng
    )

print(f"Generated synthetic cohort: {len(df):,} students | {n_schools} schools")
print(f"Retention rate: {df['retained'].mean():.1%}")
print("Missingness (survey fields):")
for column in MISSINGNESS_CONFIG:
    print(f"  {column}: {df[column].isna().mean():.1%}")

df.head()


#### Findings

The generator produces 1,000 students across 47 schools with retention near **86%**. Survey-style fields (`cash_flow_volatility`, `commute_barrier_score`, `digital_equity_access_score`, `psychosocial_support_access`) include intentional missing values at roughly 6-8%, with slightly higher missing rates at lower SES.

`retention_risk_score` and `dropout_reason` are created for simulation realism but are **excluded from modeling** (leakage / post-outcome). The saved file `elimu_match_data_v4.csv` is produced by the same logic when you run `python synthetic_data_v2.py` separately.


### Step 3: Profile the cohort

Confirm cohort size, retention rate, and leakage columns.


#### Code


In [ ]:
# Use in-memory cohort from Step 2, or load the saved file
if "df" not in globals():
    if not DATA_FILE.exists():
        raise FileNotFoundError("Run Step 2 first, or run: python synthetic_data_v2.py")
    df = pd.read_csv(DATA_FILE)

print(f"Students in cohort: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Share who stayed in school (retained=1): {df[TARGET].mean():.1%}")
print(f"Schools represented: {df['school_id'].nunique()}")

print("\nColumns excluded from modeling (leakage / post-outcome):")
for col in sorted(LEAKAGE_COLUMNS):
    print(f"  - {col}")

df.head()


#### Findings

The cohort has 1,000 students with a clear retention label. Retention is typically ~86%; class imbalance means accuracy alone is a poor metric.

Leakage columns (`retention_risk_score`, `dropout_reason`, `academic_catchup_status`) must not enter the model. The preview shows academic, health, commute, and socioeconomic fields; retention is multi-factor, not fees alone.


### Step 4: Data quality checks

Validate duplicate IDs, dropout-reason logic, and expected missingness on survey-style fields.


#### Code


In [ ]:
checks = []

# Each student should appear once
dup_ids = int(df["student_id"].duplicated().sum())
checks.append(("duplicate_student_id", dup_ids, "PASS" if dup_ids == 0 else "FAIL"))
checks.append(("retention_rate", round(df[TARGET].mean(), 3), "INFO"))

# Dropout reason should exist only when retained == 0
invalid_reason = ((df[TARGET] == 1) & df["dropout_reason"].notna()).sum()
missing_reason = ((df[TARGET] == 0) & df["dropout_reason"].isna()).sum()
checks.append(
    (
        "dropout_reason_only_when_dropped",
        f"invalid={invalid_reason}, missing={missing_reason}",
        "PASS" if invalid_reason == 0 and missing_reason == 0 else "FAIL",
    )
)

# Missingness on fields schools often do not have for every child
for col in MISSINGNESS_COLS:
    miss = df[col].isna().sum()
    checks.append((f"missing_{col}", f"{miss} ({miss / len(df):.1%})", "INFO"))

validation_df = pd.DataFrame(checks, columns=["check", "result", "status"])
validation_df


#### Findings

Structural checks should show **PASS** for duplicate IDs and dropout-reason logic. **INFO** rows on missing survey fields are expected; those gaps are handled later with missing indicators, not by dropping students.

Do not proceed to modeling if any check shows **FAIL**.


### Step 5: Class balance and retention by socioeconomic group

Check target imbalance and whether retention varies by socioeconomic index (1 = most constrained).


#### Code


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# How many dropped vs stayed
retention_counts = df[TARGET].value_counts().sort_index()
axes[0].bar(["Dropped", "Stayed"], retention_counts.values, color=["#E76F51", "#2A9D8F"])
axes[0].set_title("How many students left vs stayed")
axes[0].set_ylabel("Number of students")

# Retention rate by socioeconomic group (1 = lowest resources in this scale)
ses_ret = df.groupby("socioeconomic_status_index")[TARGET].mean().sort_index()
axes[1].plot(ses_ret.index, ses_ret.values, marker="o", color="#264653", linewidth=2)
axes[1].set_xlabel("Socioeconomic group (1 = most constrained)")
axes[1].set_ylabel("Share who stayed in school")
axes[1].set_title("Staying in school rises with socioeconomic position")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()


#### Findings

Most students are retained (left chart), so a model that always predicts "stayed" can appear accurate while missing all dropouts.

Retention increases with socioeconomic group (right chart). Model performance should be reported by group, not only in aggregate. Downstream metrics will emphasize dropout recall and AUC rather than accuracy.


### Step 6: Compare retained vs dropped students

Compare grade trend, failures, absences, commute, belonging, and income volatility between students who stayed and those who left.


#### Code


In [ ]:
compare_cols = [
    "gpa_trend",
    "failed_subjects_count",
    "health_related_absences",
    "commute_barrier_score",
    "social_integration_score",
    "cash_flow_volatility",
]
plot_df = df[[TARGET] + compare_cols].copy()
plot_df[TARGET] = plot_df[TARGET].map({0: "Left school", 1: "Stayed"})

melted = plot_df.melt(id_vars=TARGET, var_name="measure", value_name="value")
plt.figure(figsize=(12, 5))
sns.boxplot(data=melted, x="measure", y="value", hue=TARGET, showfliers=False)
plt.xticks(rotation=25, ha="right")
plt.title("Students who left tend to show worse academic, health, and access signals")
plt.tight_layout()
plt.show()


#### Findings

Students who left school show worse values on most measures: lower GPA trend, more failures, more health-related absences, longer commutes, weaker social integration, and higher cash-flow volatility.

Dropout is associated with multiple domains, which supports a multi-feature model rather than ranking on fee arrears alone. Fee balances are handled separately in the gift ledger.


### Step 7: Feature engineering

Create composite indices and interaction terms via `engineer_features()` (aligned with `preprocess_data.py`).


#### Code


In [ ]:
enriched = engineer_features(df)

print(f"New features created: {len(ENGINEERED_FEATURES)}")

corr = correlation_with_target(enriched)
print("\nStrongest relationships with staying in school (engineered features):")
display(corr.head(10).to_frame("correlation_with_stayed"))

top_feats = corr.head(8).index.tolist()
plt.figure(figsize=(7, 5))
sns.heatmap(enriched[top_feats + [TARGET]].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("How the top engineered signals relate to each other and to retention")
plt.tight_layout()
plt.show()


#### Findings

Sixteen engineered features were added. Academic risk, health burden, and economic instability indices correlate with retention in the expected direction.

Some features are correlated with each other (visible in the heatmap), which is expected when pressures cluster. A regularized linear model is a reasonable candidate alongside tree-based models. Correlations on the full cohort are exploratory; holdout evaluation follows after preprocessing.


### Step 8: Preprocessing

Remove leakage columns, add missing indicators, stratified 75/25 train/test split (seed 2026), median imputation and scaling fit on train only. Uses `preprocess()` from the project pipeline, which reads `elimu_match_data_v4.csv` from disk. *If you generated `df` in Step 2, the cell below writes it to that file first.*


#### Code


In [ ]:
# preprocess() loads elimu_match_data_v4.csv; sync disk if df came from Step 2
if "df" in globals():
    df.to_csv(DATA_FILE, index=False)

bundle = preprocess(test_size=0.25, random_state=RANDOM_STATE)

x_train = bundle["x_train"]
x_test = bundle["x_test"]
y_train = bundle["y_train"]
y_test = bundle["y_test"]
meta_test = bundle["meta_test"]
report = bundle["report"]

print("Preparation summary:")
for k in [
    "rows_train",
    "rows_test",
    "target_train_positive_rate",
    "target_test_positive_rate",
    "processed_feature_count",
]:
    print(f"  {k}: {report[k]}")

x_train.iloc[:3, :6].round(3)


#### Findings

Expect ~750 training and ~250 test rows with similar retention rates in each split. Processed feature count is ~37 after engineering, missing flags, and scaling.

All model evaluation from this point uses the held-out test set only.


### Step 9: Model comparison

Train four models: majority baseline, logistic regression, random forest, and gradient boosting. Compare test AUC, accuracy, and dropout recall.


#### Code


In [ ]:
x_train = x_train.loc[:, ~x_train.columns.str.contains("^Unnamed")]
x_test = x_test.loc[:, ~x_test.columns.str.contains("^Unnamed")]

models = {
    "Majority baseline (always stayed)": DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE),
    "Logistic regression": LogisticRegression(
        max_iter=3000, random_state=RANDOM_STATE, class_weight="balanced", C=0.1
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=400, max_depth=10, min_samples_leaf=2,
        random_state=RANDOM_STATE, class_weight="balanced",
    ),
    "Gradient boosting": HistGradientBoostingClassifier(
        max_depth=6, learning_rate=0.05, max_iter=150,
        random_state=RANDOM_STATE, class_weight="balanced",
    ),
}

rows = []
fitted = {}

for name, model in models.items():
    model.fit(x_train, y_train)
    if "Majority" in name:
        y_prob = np.full(len(y_test), float(y_train.mean()))
        y_pred = model.predict(x_test)
    else:
        y_prob = model.predict_proba(x_test)[:, 1]
        y_pred = model.predict(x_test)

    dropped_mask = y_test == DROPOUT_LABEL
    rows.append({
        "model": name,
        "test_auc": round(roc_auc_score(y_test, y_prob), 4),
        "accuracy": round((y_pred == y_test).mean(), 4),
        "recall_dropouts": round(((y_pred == 0) & dropped_mask).sum() / max(dropped_mask.sum(), 1), 4),
        "recall_stayed": round(((y_pred == 1) & (y_test == 1)).sum() / max((y_test == 1).sum(), 1), 4),
    })
    fitted[name] = {"model": model, "y_prob": y_prob, "y_pred": y_pred}

leaderboard = pd.DataFrame(rows).sort_values("test_auc", ascending=False)
leaderboard


#### Findings

The majority baseline may show the highest accuracy but zero dropout recall; it never flags a leaver.

Logistic regression and gradient boosting typically lead on AUC. Random forest often has high accuracy but low dropout recall. The next step applies a selection rule rather than picking the top accuracy.


### Step 10: Model selection

**Rule:** If top models are within 0.015 AUC, select the one with highest dropout recall; otherwise select highest AUC. Rationale: missing a dropout is costlier than extra review for a false positive.


#### Code


In [ ]:
AUC_TIE_BAND = 0.015
candidates = leaderboard[~leaderboard["model"].str.contains("Majority")].copy()
top_auc = candidates["test_auc"].max()
near_best = candidates[candidates["test_auc"] >= top_auc - AUC_TIE_BAND]
best_name = near_best.sort_values(["recall_dropouts", "test_auc"], ascending=[False, False]).iloc[0]["model"]
best = fitted[best_name]

print(f"Chosen model: {best_name}")
print(f"  Test AUC: {roc_auc_score(y_test, best['y_prob']):.3f}")
print(f"  Dropout recall: {near_best.loc[near_best['model'] == best_name, 'recall_dropouts'].values[0]:.3f}")
print("\nDetailed errors on the test set:")
print(classification_report(y_test, best["y_pred"], target_names=["left school", "stayed"]))


#### Findings

When gradient boosting and logistic regression are within 0.015 AUC, logistic regression is usually selected because it has higher dropout recall (~0.67 vs ~0.33 on this cohort).

Logistic regression is also easier to explain to school staff (coefficients / SHAP). Random forest is not selected despite higher accuracy because it under-detects dropouts. Lower precision on the dropout class is accepted because human review gates any live list.


### Step 11: ROC and confusion matrix

Visualize ranking quality (ROC) and classification errors for the selected model.


#### Code


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, best["y_prob"], ax=ax, color="#2A9D8F", name=best_name)
ax.plot([0, 1], [0, 1], "--", color="gray", label="Random guessing")
ax.set_title(f"Ranking quality: {best_name}")
ax.legend()
plt.tight_layout()
plt.show()

cm = confusion_matrix(y_test, best["y_pred"])
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Predicted left", "Predicted stayed"],
    yticklabels=["Actually left", "Actually stayed"],
)
plt.title("Where the chosen model is right and wrong")
plt.tight_layout()
plt.show()


#### Findings

The ROC curve sits above the diagonal; ranking is better than random guessing, though not perfect on synthetic data.

Confusion matrix: top-left = correctly flagged dropouts; top-right = missed dropouts (false negatives); bottom-left = false alarms (extra review); bottom-right = correctly identified stayers. This error profile is acceptable for a review queue where scores allocate support, not deny enrolment.


### Step 12: Fairness by socioeconomic group

Compute ranking AUC on each socioeconomic slice of the test set.


#### Code


In [ ]:
ses = meta_test["socioeconomic_status_index"].values
fair_rows = []
for q in sorted(np.unique(ses)):
    mask = ses == q
    y_q = y_test.values[mask]
    p_q = best["y_prob"][mask]
    auc = roc_auc_score(y_q, p_q) if len(np.unique(y_q)) > 1 else np.nan
    fair_rows.append({
        "socioeconomic_group": int(q),
        "students_in_test": int(mask.sum()),
        "share_who_stayed": round(float(y_q.mean()), 3),
        "ranking_auc": auc,
    })

fair_df = pd.DataFrame(fair_rows)
display(fair_df)

plt.figure(figsize=(7, 4))
sns.barplot(data=fair_df, x="socioeconomic_group", y="ranking_auc", color="#E9C46A")
plt.axhline(0.5, color="gray", linestyle="--", label="No better than chance")
plt.ylim(0, 1)
plt.xlabel("Socioeconomic group (1 = most constrained)")
plt.ylabel("Ranking AUC on test slice")
plt.title("Model ranking is weaker where retention is already very high")
plt.tight_layout()
plt.show()


#### Findings

AUC is often lower in higher socioeconomic groups where almost all students stay (little variation to rank). Lower groups show more dropout variation and more stable AUC.

This does not by itself prove bias, but it requires termly fairness reporting by SES and gender before publishing ranked lists. Retrain or adjust thresholds if slice metrics worsen on partner data.


### Step 13: Global feature importance (SHAP)

Show which features drive risk scores for operations staff. Uses precomputed SHAP output if available; otherwise model coefficients.


#### Code


In [ ]:
shap_csv = ROOT / "shap_outputs" / "shap_global_importance.csv"
if shap_csv.exists():
    shap_imp = pd.read_csv(shap_csv).head(12)
    if "feature_clean" not in shap_imp.columns:
        shap_imp["feature_clean"] = shap_imp["feature"]
    plt.figure(figsize=(9, 5))
    sns.barplot(data=shap_imp, x="mean_abs_shap", y="feature_clean", color="#264653")
    plt.xlabel("Average influence on risk score")
    plt.title("What drives dropout risk across the cohort")
    plt.tight_layout()
    plt.show()
else:
    print("Full SHAP charts: run python shap_analysis.py after modeling_phase.py")
    m = best["model"]
    if hasattr(m, "coef_"):
        imp = pd.Series(np.abs(m.coef_).ravel(), index=x_test.columns).sort_values(ascending=False).head(12)
        display(imp.to_frame("abs_coefficient"))


#### Findings

Top drivers are typically health burden, academic performance combined with SES, commute/barrier measures, and belonging, not fee arrears alone.

Operations should route non-fee needs (health, tutoring) through school/partner channels. Explainability is shown to staff on the analytics surface, not on the helper gift screen. Run `shap_analysis.py` for full beeswarm and dependence plots.


### Step 14: Intervention queue size

Read intervention summary counts if `intervention_matrix.py` has been run (shows how many students route to each support type).


#### Code


In [ ]:
intervention_path = ROOT / "intervention_outputs" / "intervention_summary.csv"
if intervention_path.exists():
    inter = pd.read_csv(intervention_path)
    display(inter.sort_values("students", ascending=False))
    fee_row = inter.loc[inter["intervention"] == "School Fee Support"]
    if not fee_row.empty:
        n_fee = int(fee_row["students"].iloc[0])
        print(f"\nFee-support lane: about {n_fee} students from {len(df):,} in the cohort.")
        print("That scale is intentional: a pilot queue, not a national roll call.")
else:
    print("Build routing tables with: python cluster_personas.py && python intervention_matrix.py")


#### Findings

School Fee Support typically covers ~280 students from 1,000, a manageable pilot queue. Other interventions (tutoring, health, digital) have separate counts and are owned by schools/partners.

If the CSV is missing, run `cluster_personas.py` and `intervention_matrix.py` to generate routing tables.


## 15. Conclusion and recommendation

### Summary
1. Retention is imbalanced (~86% stay); use AUC and dropout recall, not accuracy alone.
2. Risk is multi-factor (academic, health, access, SES); fee arrears belong in the ledger, not as the sole model input.
3. Logistic regression is selected when AUC is tied with boosting; higher dropout recall and easier to explain.
4. AUC varies by socioeconomic slice; require termly fairness monitoring.
5. Fee support is one intervention lane; health and academic drivers imply non-fee routing for operations.

### Recommendation
- Deploy logistic regression for an eight-school pilot with human review before any live list.
- Monitor pilot KPIs: gift coverage, settlement integrity, fairness cadence, retention among helped students.
- Retrain on partner school data before scaling beyond the pilot.

### Reproduce full pipeline
```text
python synthetic_data_v2.py
python preprocess_data.py
python modeling_phase.py
python shap_analysis.py
python cluster_personas.py
python intervention_matrix.py
```

**Demo:** https://jesyldah.github.io/ElimuMatch/
